In [0]:
-- Source data preview
CREATE TEMPORARY VIEW source_data AS
SELECT * FROM firstcatalog.firstschema.electric_vehicle_population_data

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW dim_city
AS
SELECT
  md5(
    concat(
      coalesce(City, ''),
      coalesce(County, ''),
      coalesce(State, ''),
      coalesce(`Postal Code`, cast(NULL as STRING)),
      coalesce(`Legislative District`, cast(NULL as STRING))
    )
  ) as city_id,
  City as city,
  County as county,
  State as state,
  `Postal Code` as postal_code,
  `Legislative District` as legislative_district
FROM
  firstcatalog.firstschema.electric_vehicle_population_data
WHERE
  md5(
    concat(
      coalesce(City, ''),
      coalesce(County, ''),
      coalesce(State, ''),
      coalesce(`Postal Code`, cast(NULL as STRING)),
      coalesce(`Legislative District`, cast(NULL as STRING))
    )
  ) IS NOT NULL
GROUP BY
  City,
  County,
  State,
  `Postal Code`,
  `Legislative District`

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW dim_year
AS
SELECT 
  md5(concat(coalesce(`Model Year`, ''))) as year_id,
  `Model Year` as model_year
FROM 
  firstcatalog.firstschema.electric_vehicle_population_data
WHERE 
  md5(concat(coalesce(`Model Year`, ''))) IS NOT NULL
GROUP BY 
  `Model Year`

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW dim_electric_utility
AS
SELECT
  md5(concat(coalesce(`Electric Utility`, ''))) as electric_utility_id,
  `Electric Utility` as electric_utility
FROM
  firstcatalog.firstschema.electric_vehicle_population_data
WHERE
  md5(concat(coalesce(`Electric Utility`, ''))) IS NOT NULL
GROUP BY
  `Electric Utility`

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW dim_model
AS
SELECT 
  md5(concat(coalesce(Model, ''), coalesce(Make, ''))) AS model_id,
  Model,
  Make
FROM 
  firstcatalog.firstschema.electric_vehicle_population_data
WHERE 
  md5(concat(coalesce(Model, ''), coalesce(Make, ''))) IS NOT NULL
GROUP BY 
  Model, 
  Make

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW dim_vehicle_type
AS
SELECT
  md5(concat(coalesce(`Electric Vehicle Type`, ''))) as vehicle_type_id,
  `Electric Vehicle Type` as electric_vehicle_type
FROM
  firstcatalog.firstschema.electric_vehicle_population_data
WHERE
  md5(concat(coalesce(`Electric Vehicle Type`, ''))) IS NOT NULL
GROUP BY
  `Electric Vehicle Type`;


CREATE OR REFRESH MATERIALIZED VIEW fact_vehicle
AS
SELECT
  md5(
    concat(
      coalesce(a.Clean_Alternative_Fuel_Vehicle_CAFV_Eligibility, ''),
      coalesce(a.`Electric Range`, cast(null as STRING)),
      coalesce(a.`Base MSRP`, cast(null as STRING)),
      coalesce(a.`DOL Vehicle ID`, ''),
      coalesce(a.`Vehicle Location`, '')
    )
  ) as vehicle_id,
  c.city_id,
  y.year_id,
  m.model_id,
  u.electric_utility_id,
  v.vehicle_type_id,
  a.Clean_Alternative_Fuel_Vehicle_CAFV_Eligibility,
  a.`Electric Range` as electric_range,
  a.`Base MSRP` as base_msrp,
  a.`DOL Vehicle ID` as dol_vehicle_id,
  a.`Vehicle Location` as vehicle_location
FROM
  firstcatalog.firstschema.electric_vehicle_population_data a
  LEFT JOIN dim_city c ON md5(
    concat(
      coalesce(a.City, ''),
      coalesce(a.County, ''),
      coalesce(a.State, ''),
      coalesce(a.`Postal Code`, cast(NULL as STRING)),
      coalesce(a.`Legislative District`, cast(NULL as STRING))
    )
  ) = c.city_id
  LEFT JOIN dim_year y ON md5(concat(coalesce(a.`Model Year`, ''))) = y.year_id
  LEFT JOIN dim_model m ON md5(concat(coalesce(a.Model, ''), coalesce(a.Make, ''))) = m.model_id
  LEFT JOIN dim_electric_utility u ON md5(concat(coalesce(a.`Electric Utility`, ''))) = u.electric_utility_id
  LEFT JOIN dim_vehicle_type v ON md5(concat(coalesce(a.`Electric Vehicle Type`, ''))) = v.vehicle_type_id
WHERE
  md5(
    concat(
      coalesce(a.Clean_Alternative_Fuel_Vehicle_CAFV_Eligibility, ''),
      coalesce(a.`Electric Range`, cast(null as STRING)),
      coalesce(a.`Base MSRP`, cast(null as STRING)),
      coalesce(a.`DOL Vehicle ID`, ''),
      coalesce(a.`Vehicle Location`, '')
    )
  ) IS NOT NULL
GROUP BY ALL